In [8]:
import os
import sys
from pathlib import Path
print('cwd=', os.getcwd())
print('python=', sys.executable)
print('exists plantuml?', Path('/usr/bin/plantuml').exists())
print('which plantuml:', os.popen('command -v plantuml').read().strip())
print('java version:', os.popen('java -version 2>&1').read().splitlines()[0])
print('workspace file count=', sum(1 for p in Path('.').glob('**/*') if p.is_file()))
print('sample files:')
for p in sorted(Path('.').glob('**/*'))[:20]:
    if p.is_file():
        print(p)


cwd= /workspaces/2025-p1a-inf-nis-filipvyroubal
python= /workspaces/2025-p1a-inf-nis-filipvyroubal/.venv/bin/python
exists plantuml? False
which plantuml: 
java version: openjdk version "25.0.2" 2026-01-20 LTS
workspace file count= 9614
sample files:
.git/COMMIT_EDITMSG
.git/FETCH_HEAD
.git/HEAD
.git/ORIG_HEAD
.git/config
.git/description
.git/hooks/applypatch-msg.sample
.git/hooks/commit-msg.sample
.git/hooks/fsmonitor-watchman.sample
.git/hooks/post-checkout
.git/hooks/post-commit
.git/hooks/post-merge
.git/hooks/post-update.sample
.git/hooks/pre-applypatch.sample
.git/hooks/pre-commit.sample
.git/hooks/pre-merge-commit.sample
.git/hooks/pre-push
.git/hooks/pre-push.sample


In [7]:
import urllib.request
import subprocess
from pathlib import Path

root = Path('.')
jar = root / 'plantuml.jar'
if not jar.exists():
    print('Downloading PlantUML jar...')
    urllib.request.urlretrieve(
        'https://github.com/plantuml/plantuml/releases/download/v1.2024.3/plantuml.jar',
        jar
    )
    print('Downloaded', jar)
else:
    print('PlantUML jar already present:', jar)

for puml in sorted(root.glob('diagrams/*.puml')):
    print('Rendering', puml.name)
    result = subprocess.run(['java', '-jar', str(jar), '-tsvg', str(puml)], capture_output=True, text=True)
    print('returncode=', result.returncode)
    if result.stdout:
        print('stdout:', result.stdout)
    if result.stderr:
        print('stderr:', result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'PlantUML render failed for {puml.name}')
print('SVG generation done.')


PlantUML jar already present: plantuml.jar
Rendering use_case_diagram_new.puml
returncode= 0
stderr: java.io.IOException: Cannot run program "/opt/local/bin/dot": Exec failed, error: 2 (No such file or directory) 
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1112)
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1046)
	at java.base/java.lang.Runtime.exec(Runtime.java:605)
	at net.sourceforge.plantuml.dot.ProcessRunner$MainThread.startThreads(ProcessRunner.java:165)
	at net.sourceforge.plantuml.dot.ProcessRunner$MainThread.runJob(ProcessRunner.java:125)
	at net.sourceforge.plantuml.api.TimeoutExecutor$MyThread.run(TimeoutExecutor.java:81)
Caused by: java.io.IOException: Exec failed, error: 2 (No such file or directory) 
	at java.base/java.lang.ProcessImpl.forkAndExec(Native Method)
	at java.base/java.lang.ProcessImpl.<init>(ProcessImpl.java:300)
	at java.base/java.lang.ProcessImpl.start(ProcessImpl.java:231)
	at java.base/java.lang.ProcessBuilder.star

In [9]:
import os
print('which dot:', os.popen('which dot').read().strip())
print('dot exists at /opt/local/bin/dot?', os.path.exists('/opt/local/bin/dot'))


which dot: 
dot exists at /opt/local/bin/dot? False


In [10]:
import zlib
import urllib.request
from pathlib import Path

base_url = 'https://www.plantuml.com/plantuml/svg/'

alphabet = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz-_"

def encode64(data):
    res = []
    for i in range(0, len(data), 3):
        if i+2 == len(data):
            b1, b2 = data[i], data[i+1]
            res.append(alphabet[b1 >> 2])
            res.append(alphabet[((b1 & 0x3) << 4) | (b2 >> 4)])
            res.append(alphabet[(b2 & 0xF) << 2])
            res.append(alphabet[0])
        elif i+1 == len(data):
            b1 = data[i]
            res.append(alphabet[b1 >> 2])
            res.append(alphabet[(b1 & 0x3) << 4])
            res.append(alphabet[0])
            res.append(alphabet[0])
        else:
            b1, b2, b3 = data[i], data[i+1], data[i+2]
            res.append(alphabet[b1 >> 2])
            res.append(alphabet[((b1 & 0x3) << 4) | (b2 >> 4)])
            res.append(alphabet[((b2 & 0xF) << 2) | (b3 >> 6)])
            res.append(alphabet[b3 & 0x3F])
    return ''.join(res)


def plantuml_encode(text):
    compressed = zlib.compress(text.encode('utf-8'))
    return encode64(compressed[2:-4])

for puml_path in sorted(Path('diagrams').glob('*.puml')):
    text = puml_path.read_text(encoding='utf-8')
    code = plantuml_encode(text)
    url = base_url + code
    print('Fetching SVG for', puml_path.name)
    svg_data = urllib.request.urlopen(url, timeout=30).read().decode('utf-8')
    svg_path = puml_path.with_suffix('.svg')
    svg_path.write_text(svg_data, encoding='utf-8')
    print('Saved', svg_path)


Fetching SVG for use_case_diagram_new.puml


HTTPError: HTTP Error 403: Forbidden

In [13]:
import zlib
import urllib.request
from pathlib import Path

base_url = 'https://www.plantuml.com/plantuml/svg/'

alphabet = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz-_"

def encode64(data):
    res = []
    for i in range(0, len(data), 3):
        if i+2 == len(data):
            b1, b2 = data[i], data[i+1]
            res.append(alphabet[b1 >> 2])
            res.append(alphabet[((b1 & 0x3) << 4) | (b2 >> 4)])
            res.append(alphabet[(b2 & 0xF) << 2])
            res.append(alphabet[0])
        elif i+1 == len(data):
            b1 = data[i]
            res.append(alphabet[b1 >> 2])
            res.append(alphabet[(b1 & 0x3) << 4])
            res.append(alphabet[0])
            res.append(alphabet[0])
        else:
            b1, b2, b3 = data[i], data[i+1], data[i+2]
            res.append(alphabet[b1 >> 2])
            res.append(alphabet[((b1 & 0x3) << 4) | (b2 >> 4)])
            res.append(alphabet[((b2 & 0xF) << 2) | (b3 >> 6)])
            res.append(alphabet[b3 & 0x3F])
    return ''.join(res)


def plantuml_encode(text):
    compressed = zlib.compress(text.encode('utf-8'))
    return encode64(compressed[2:-4])

for puml_path in sorted(Path('diagrams').glob('*.puml')):
    text = puml_path.read_text(encoding='utf-8')
    code = plantuml_encode(text)
    url = base_url + code
    print('Fetching SVG for', puml_path.name)
    request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    svg_data = urllib.request.urlopen(request, timeout=30).read().decode('utf-8')
    svg_path = puml_path.with_suffix('.svg')
    svg_path.write_text(svg_data, encoding='utf-8')
    print('Saved', svg_path)


Fetching SVG for use_case_diagram_new.puml
Saved diagrams/use_case_diagram_new.svg


In [12]:
import os
from pathlib import Path

extra = Path('diagrams/usecase_diagram_new.svg')
if extra.exists():
    extra.unlink()
    print('Removed duplicate file', extra)
else:
    print('No duplicate file to remove')


Removed duplicate file diagrams/usecase_diagram_new.svg
